# Topic: The with Statement - Resource leak prevention, file descriptors, and exception safety
 
## 1. THE PROBLEM: MANUAL CLOSE & FILE LEAKS
- **The Risk**: Opening a file using `f = open(...)` and calling `f.close()` manually is fragile:
  - If an exception occurs in the lines between `open` and `close`, the `close` statement is bypassed.
  - The file descriptor remains open in the OS, leaking resources.
  - **Too many open files**: Operating systems limit the number of active file descriptors a single process can open. Accumulating leaks eventually raises `OSError: [Errno 24] Too many open files`.
 
## 2. THE SOLUTION: THE WITH STATEMENT
- Introduced in PEP 343 (Python 2.5).
- **with statement**:
  - Acts as a context manager wrapping file streams.
  - Guarantees that `f.close()` is invoked automatically as soon as execution leaves the block, regardless of whether it exits normally or raises a runtime exception.
 
## 3. EQUIVALENT BLOCK
- The `with` statement:
  ```python
  with open("file.txt") as f:
      data = f.read()
  ```
  Is syntactically equivalent to the manual `try...finally` block:
  ```python
  f = open("file.txt")
  try:
      data = f.read()
  finally:
      f.close()
  ```
 
---


In [ ]:
import os

with_demo_file = "with_demo.txt"

# 1. Verification of Exception Safety
print("--- Context Manager Exception Safety ---")

# Let's write a file first
with open(with_demo_file, "w", encoding="utf-8") as f:
    f.write("Some text info.")

# Global file object reference
saved_f = None

try:
    with open(with_demo_file, "r", encoding="utf-8") as f:
        saved_f = f  # Keep a reference to verify state
        print(f"Inside with block - Is Closed? {f.closed}")  # Expected: False
        print(" -> Raising an intentional error inside block...")
        raise ZeroDivisionError("Intentional Error!")
except ZeroDivisionError:
    print(" -> Exception caught outside with block.")

# Even though an exception was raised, Python closed the file automatically!
print(f"Outside with block - Is Closed? {saved_f.closed}")  # Expected: True



In [ ]:
# 2. Replicating the equivalent try...finally block manually
print("\n--- Manual try...finally Equivalent ---")
f_manual = open(with_demo_file, "r", encoding="utf-8")
try:
    print(f"Inside manual try - Is Closed? {f_manual.closed}")  # Expected: False
    data = f_manual.read()
finally:
    # This block executes even if try raises errors
    f_manual.close()
    print(" -> Finally block: file closed manually.")
    
print(f"Outside manual block - Is Closed? {f_manual.closed}")  # Expected: True



In [ ]:
# 3. Clean up
if os.path.exists(with_demo_file):
    os.remove(with_demo_file)
